# Curating systems for SLiM simulations

In [2]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import shutil

os.chdir('/home/fknudsen/calvados-slim')

from src import curation, utils

In [3]:
# Setting SMOG bin path, for SCM calculations
os.environ["SMOGBIN"] = os.path.expanduser("~/software/smog-2.5/bin")

In [4]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

# Loading configurations
with open('data/configurations/conditions.json') as file:
    conditions = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)
with open('data/configurations/go_scans.json') as file:
    go_scans = json.load(file)
slurm_scripts_dir = 'data/configurations/slurm'

In [5]:
# Curating input files for systems
for system in tqdm(systems):
    system_dir = os.path.join(systems_dir, system['ID'])
    if os.path.exists(system_dir):
        continue

    assert system['Box sides'][0] < 100, "This is a failsafe for accidental box dimensions over 100 nm"

    # Getting missing CIF files
    if not os.path.exists(system['SLiMs'][0]['PDB']):
        path, extension = os.path.splitext(system['SLiMs'][0]['PDB'])
        pdb_id = os.path.basename(path)
        if extension == '.cif' and len(pdb_id) == 4:
            utils.download_cif(pdb_id, system['SLiMs'][0]['PDB'])

    try:
        curation.System.curate(system, output_dir=systems_dir)
    except Exception as e:
        print(f"Curation failed for system '{system['ID']}': {e}")
        shutil.rmtree(os.path.join(system_dir, system['ID'])) 

  0%|          | 0/13 [00:00<?, ?it/s]/home/fknudsen/software/miniconda3/envs/calvados/lib/python3.10/site-packages/MDAnalysis/lib/util.py:708: RuntimeWarning: Constructed NamedStream from a NamedStream
  warnings.warn(
100%|██████████| 13/13 [00:07<00:00,  1.70it/s]
